# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import geopandas as gpd
from shapely import affinity, Polygon, Point
import folium
# from folium import plugins
# from folium.plugins import HeatMap
import osmnx as ox
from math import radians, cos, sin, asin, sqrt
import requests
import urllib
import networkx as nx
import pickle
from scipy import stats

Image.MAX_IMAGE_PIXELS = None

from dimod import ConstrainedQuadraticModel, Binary, quicksum
from dwave.system import LeapHybridCQMSampler

# import branca.colormap as cm
# from matplotlib.colors import Normalize
from branca.element import Template, MacroElement
from copy import deepcopy
import random

## Functions

In [ ]:
# --------------------------------------------
# MISC. FUNCTIONS
# --------------------------------------------
def get_poly_sqr_from_bounds(active_zone_bounds):
    
    active_zone_ll = Polygon([[active_zone_bounds[0][0],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][0]]])

    return active_zone_ll

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points 
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 3956 #6371 # Radius of earth in kilometers. Use 3956 for miles. Determines return value units.
    return c * r

In [ ]:
project_title = "CA Tree Clumps Fire"

In [ ]:
# --------------------------------------------
# ORIGINAL D-WAVE FUNCTIONS
# --------------------------------------------


# Original function for building constrained quadratic model
def build_cqm(G):
    """ Build the CQM for the problem instance."""

    # Two groups (cases 0, 1) and one separator group (case 2)
    num_groups = 3

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]

    # Set objective for CQM
    cqm.set_objective(quicksum(vars[i][2] for i in range(len(vars))))

    # Add constraint to make variables discrete
    for v in range(len(vars)):
        cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups)])

    # Add constraint to CQM: |G1|=|G2|
    g1 = [vars[i][0] for i in range(len(vars))]
    g2 = [vars[i][1] for i in range(len(vars))]
    cqm.add_constraint(quicksum(g1) - quicksum(g2) <= 1)
    cqm.add_constraint(quicksum(g1) - quicksum(g2) >= -1)

    # Add constraint to CQM: e(G1, G2) = 0
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.add_constraint(quicksum(edge_sum) == 0, label='cross edges')

    return cqm

def run_cqm_and_collect_solutions(cqm, sampler):
    """ Send the CQM to the sampler and return the best sample found."""

    # Initialize the solver
    print("\nSending to the solver...")
    
    # Solve the CQM problem using the solver
    sampleset = sampler.sample_cqm(cqm, label=project_title)

    # Get the first feasible solution
    feasible_sampleset = sampleset.filter(lambda d: d.is_feasible)
    if len(feasible_sampleset) == 0:
        print("\nNo feasible solution found. Returning best infeasible solution.")
        return sampleset.first.sample

    return feasible_sampleset.first.sample

def process_sample(G, sample):
    """ Interpret the CQM solution in terms of the partitioning problem."""

    # Display results to user
    group_1 = []
    group_2 = []
    sep_group = []
    results = [[],[],[]]
    for key, val in sample.items(): # key is of form "x_{node}_{group #}" and val is 0 or 1
        if val == 1:
            v = key.split("_")
            results[int(v[-1])].append(int(v[1]))

    group_1 = results[0]
    group_2 = results[1]
    sep_group = results[2]

    # Display best result
    print("\nPartition Found:")
    print("\tGroup 1: \tSize", len(group_1))
    print("\tGroup 2: \tSize", len(group_2))
    print("\tSeparator: \tSize", len(sep_group))

    print("\nSeparator Fraction: \t", len(sep_group)/len(G.nodes()))

    # Determines if there are any edges directly between the large groups
    illegal_edges = [(u, v) for u, v in G.edges if (sample[f'x_{u}_{0}']*sample[f'x_{v}_{1}'] == 1 or sample[f'x_{u}_{1}']*sample[f'x_{v}_{0}'] == 1)]

    print("\nNumber of illegal edges:\t", len(illegal_edges))

    return group_1, group_2, sep_group, illegal_edges

def visualize_results(G, group_1, group_2, sep_group, illegal_edges, save = False):
    """ Visualize the partition."""

    print("\nVisualizing output...")

    G1 = G.subgraph(group_1)
    G2 = G.subgraph(group_2)
    SG = G.subgraph(sep_group)

    pos_1 = nx.random_layout(G1, center=(-5,0))
    pos_2 = nx.random_layout(G2, center=(5,0))
    pos_sep = nx.random_layout(SG, center=(0,0))
    pos = {**pos_1, **pos_2, **pos_sep}

    nx.draw_networkx_nodes(G, pos_1, node_size=10, nodelist=group_1, node_color='#17bebb', edgecolors='k')
    nx.draw_networkx_nodes(G, pos_2, node_size=10, nodelist=group_2, node_color='#2a7de1', edgecolors='k')
    nx.draw_networkx_nodes(G, pos_sep, node_size=10, nodelist=sep_group, node_color='#f37820', edgecolors='k')

    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), style='solid', edge_color='#808080')
    nx.draw_networkx_edges(G, pos, edgelist=illegal_edges, style='solid')

    # plt.draw()
    if save:
        output_name = 'separator.png'
        plt.savefig(output_name)
        print("\tOutput stored in", output_name)
    plt.show()
    # plt.close()
    




In [ ]:
# --------------------------------------------
# MODIFIED D-WAVE FUNCTIONS
# --------------------------------------------


def build_cqm_tol(G, tolerance = 'default'):
    """ Build the CQM for the problem instance."""
    # Original function for building constrained quadratic model, group size constraint with tolerance
    
    # Two groups (cases 0, 1) and one separator group (case 2)
    num_groups = 3

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]

    # Set objective for CQM
    cqm.set_objective(quicksum(vars[i][2] for i in range(len(vars))))

    # Add constraint to make variables discrete
    for v in range(len(vars)):
        cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups)])

    # Add constraint to CQM: | |G1|-|G2| | <= tolerance
    if tolerance == 'default':
        tolerance = (len(G.nodes))/40
    g1 = [vars[i][0] for i in range(len(vars))]
    g2 = [vars[i][1] for i in range(len(vars))]
    cqm.add_constraint(quicksum(g1) - quicksum(g2) <= tolerance)
    cqm.add_constraint(quicksum(g1) - quicksum(g2) >= tolerance)

    # Add constraint to CQM: e(G1, G2) = 0
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.add_constraint(quicksum(edge_sum) == 0, label='cross edges')

    return cqm

# --------------------------------------------
# Using Separator Bounds and preassignment
# --------------------------------------------
def preassign(G, split, bound1, bound2):
    # Returns nodes outside of bound1 and bound2 as two lists
    if bound1 >= bound2:
        raise Exception("bound1 must be smaller than bound2")
    
    pre_G1 = []
    pre_G2 = []
           
    if split in ['vertical','longitude']:
        for node in G.nodes():
            c = centroids[node]
            if c.x <= bound1:
                pre_G1.append(node)
            elif c.x >= bound2:
                pre_G2.append(node)
                
    elif split in ['horizontal','latitude']:
        for node in G.nodes():
            c = centroids[node]
            if c.y <= bound1:
                pre_G1.append(node)
            elif c.y >= bound2:
                pre_G2.append(node)
    
    return pre_G1, pre_G2

def build_cqm_pa(G, pre_G1, pre_G2):
    """ Build the CQM for the problem instance."""
    # CQM with objective being minimizing the number of separators, but includes preassignment
    # Includes size constraint on groups
    
    # Two groups (cases 0, 1) and one separator group (case 2)
    num_groups = 3

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]

    # Set objective for CQM
    cqm.set_objective(quicksum(vars[i][2] for i in range(len(vars))))

    # Add constraint to make variables discrete
    for v in range(len(vars)):
        cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups)])

    # Add constraint to CQM: |G1|=|G2|
    g1 = [vars[i][0] for i in range(len(vars))]
    g2 = [vars[i][1] for i in range(len(vars))]
    cqm.add_constraint(quicksum(g1) - quicksum(g2) <= 1)
    cqm.add_constraint(quicksum(g1) - quicksum(g2) >= -1)

    # Add constraint to CQM: e(G1, G2) = 0
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.add_constraint(quicksum(edge_sum) == 0, label='cross edges')
    
    # Fix variables from preassignment
    for node in pre_G1:
        cqm.fix_variables({f'x_{node}_0': 1, f'x_{node}_1': 0, f'x_{node}_2': 0})
    for node in pre_G2:
        cqm.fix_variables({f'x_{node}_0': 0, f'x_{node}_1': 1, f'x_{node}_2': 0})

    return cqm

def build_cqm_pa_tol(G, pre_G1, pre_G2, tolerance = 'default'):
    """ Build the CQM for the problem instance."""
    # CQM with objective being minimizing the number of separators, but includes preassignment and a tolerance on group size
    
    # Two groups (cases 0, 1) and one separator group (case 2)
    num_groups = 3

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]

    # Set objective for CQM
    cqm.set_objective(quicksum(vars[i][2] for i in range(len(vars))))

    # Add constraint to make variables discrete
    for v in range(len(vars)):
        cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups)])

    # Add constraint to CQM: | |G1| - |G2| | < = tolerance
    if tolerance == 'default':
        tolerance = (len(G.nodes))/40
    g1 = [vars[i][0] for i in range(len(vars))]
    g2 = [vars[i][1] for i in range(len(vars))]
    cqm.add_constraint(quicksum(g1) - quicksum(g2) <= tolerance)
    cqm.add_constraint(quicksum(g1) - quicksum(g2) >= -tolerance)

    # Add constraint to CQM: e(G1, G2) = 0
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.add_constraint(quicksum(edge_sum) == 0, label='cross edges')
    
    # Fix variables from preassignment
    for node in pre_G1:
        cqm.fix_variables({f'x_{node}_0': 1, f'x_{node}_1': 0, f'x_{node}_2': 0})
    for node in pre_G2:
        cqm.fix_variables({f'x_{node}_0': 0, f'x_{node}_1': 1, f'x_{node}_2': 0})

    return cqm

def build_cqm_preassigned(G, size, pre_G1, pre_G2):
    """ Build the CQM for the problem instance."""
    # CQM with objective being minimizing the number of cross edges, and with the number of separators given (size).
    # Also includes preassignment.

    # Two groups (cases 0, 1) and one separator group (case 2)
    num_groups = 3

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]

    # Set objective for CQM: minimize e(G1, G2)
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.set_objective(quicksum(edge_sum))

    # Add constraint to make variables discrete
    for v in range(len(vars)):
        cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups)])
        

    # Add constraint to CQM: |G3|=size
    g3 = [vars[i][2] for i in range(len(vars))]
    cqm.add_constraint(quicksum(g3) == size)
    
    
    # Fix variables from preassignment
    for node in pre_G1:
        cqm.fix_variables({f'x_{node}_0': 1, f'x_{node}_1': 0, f'x_{node}_2': 0})
    for node in pre_G2:
        cqm.fix_variables({f'x_{node}_0': 0, f'x_{node}_1': 1, f'x_{node}_2': 0})
        
    return cqm

def process_sample_preassigned(G, sample, pre_G1, pre_G2):
    """ Interpret the CQM solution in terms of the partitioning problem."""
    # Display results to user
    group_1 = []
    group_2 = []
    sep_group = []
    results = [[],[],[]]
    for key, val in sample.items(): # key is of form "x_{node}_{group #}" and val is 0 or 1
        if val == 1:
            v = key.split("_")
            results[int(v[-1])].append(int(v[1]))
    
    # The sample does not include fixed variables, so you must manually add the preassigned ones
    group_1 = pre_G1+results[0]
    group_2 = pre_G2+results[1]
    sep_group = results[2]

    # Display best result
    print("\nPartition Found:")
    print("\tGroup 1: \tSize", len(group_1))
    print("\tGroup 2: \tSize", len(group_2))
    print("\tSeparator: \tSize", len(sep_group))

    print("\nSeparator Fraction: \t", len(sep_group)/len(G.nodes()))

    # Determines if there are any edges directly between the large groups
    # The sample does not include fixed variables, so you must manually include preassigned variable values
    # illegal_edges = [(u, v) for u, v in G.edges if (sample[f'x_{u}_{0}']*sample[f'x_{v}_{1}'] == 1 or sample[f'x_{u}_{1}']*sample[f'x_{v}_{0}'] == 1)]
    illegal_edges = []
    for u, v in G.edges:
        if u in pre_G1:
            xu0 = 1
            xu1 = 0
        elif u in pre_G2:
            xu0 = 0
            xu1 = 1 
        else:
            xu0 = sample[f'x_{u}_{0}']
            xu1 = sample[f'x_{u}_{1}']
        
        if v in pre_G1:
            xv0 = 1
            xv1 = 0
        elif v in pre_G2:
            xv0 = 0
            xv1 = 1 
        else:
            xv0 = sample[f'x_{v}_{0}']
            xv1 = sample[f'x_{v}_{1}']
        
        if xu0*xv1 == 1 or xu1*xv0 == 1:
            illegal_edges.append((u,v))
            
    print("\nNumber of illegal edges:\t", len(illegal_edges))

    return group_1, group_2, sep_group, illegal_edges

# --------------------------------------------
# Using Fixed Separators and preassignment
# --------------------------------------------

def build_cqm_from_sep_preassigned(G, separators, pre_G1, pre_G2):
    """ Build the CQM for the problem instance."""

    # Two groups (cases 0, 1) and one separator group (case 2)
    # We will set case 2 using given separators, so really there are just two groups.
    num_groups = 2

    # Initialize the CQM object
    print("\nBuilding CQM...")
    cqm = ConstrainedQuadraticModel()

    # Build the CQM starting by creating variables
    # Separators fixed, so just two group variables per non-separator node
    vars = [[Binary(f'x_{name}_{i}') for i in range(num_groups)] for name in G.nodes()]
    
    # Set objective for CQM: minimize e(G1, G2) = 0
    edge_sum = []
    for a, b in G.edges():
        if a != b:
            edge_sum.append(vars[a][0]*vars[b][1]+vars[a][1]*vars[b][0])
    cqm.set_objective(quicksum(edge_sum))
    

    # Add constraint to make variables discrete
    # for v in range(len(vars)):
    #     cqm.add_discrete([f'x_{v}_{i}' for i in range(num_groups - 1)])
    for name in G.nodes():
        cqm.add_discrete([f'x_{name}_{i}' for i in range(num_groups)])
    
    
    # Fix variables from pre-processing and separators
    for node in pre_G1:
        cqm.fix_variables({f'x_{node}_0': 1, f'x_{node}_1': 0})
    for node in pre_G2:
        cqm.fix_variables({f'x_{node}_0': 0, f'x_{node}_1': 1})
    for node in separators:
        cqm.fix_variables({f'x_{node}_0': 0, f'x_{node}_1': 0})
    
    return cqm

def process_sample_with_sep_preassigned(G, sample, separators, pre_G1, pre_G2):
    """ Interpret the CQM solution in terms of the partitioning problem."""

    # Display results to user
    group_1 = []
    group_2 = []
    # sep_group = []
    results = [[],[],[]]
    for key, val in sample.items(): # key is of form "x_{node}_{group #}" and val is 0 or 1
        if val == 1:
            v = key.split("_")
            results[int(v[-1])].append(int(v[1]))
            
    # The sample does not include fixed variables, so you must manually add the preassigned ones and the separators
    group_1 = pre_G1+results[0]
    group_2 = pre_G2+results[1]
    sep_group = separators

    # Display best result
    print("\nPartition Found:")
    print("\tGroup 1: \tSize", len(group_1))
    print("\tGroup 2: \tSize", len(group_2))
    print("\tSeparator: \tSize", len(sep_group))

    print("\nSeparator Fraction: \t", len(sep_group)/len(G.nodes()))

    # Determines if there are any edges directly between the large groups
    # The sample does not include fixed variables, so you must manually include preassigned variable values
    # illegal_edges = [(u, v) for u, v in G.edges if (sample[f'x_{u}_{0}']*sample[f'x_{v}_{1}'] == 1 or sample[f'x_{u}_{1}']*sample[f'x_{v}_{0}'] == 1)]
    illegal_edges = []
    for u, v in G.edges:
        if u not in separators and v not in separators:
            if u in pre_G1:
                xu0 = 1
                xu1 = 0
            elif u in pre_G2:
                xu0 = 0
                xu1 = 1 
            else:
                xu0 = sample[f'x_{u}_{0}']
                xu1 = sample[f'x_{u}_{1}']

            if v in pre_G1:
                xv0 = 1
                xv1 = 0
            elif v in pre_G2:
                xv0 = 0
                xv1 = 1 
            else:
                xv0 = sample[f'x_{v}_{0}']
                xv1 = sample[f'x_{v}_{1}']

            if xu0*xv1 == 1 or xu1*xv0 == 1:
                illegal_edges.append((u,v))
                print('illegal edge: {}, {}'.format(u,v))
            
    print("\nNumber of illegal edges:\t", len(illegal_edges))

    return group_1, group_2, illegal_edges

In [ ]:
# --------------------------------------------
# MAP FUNCTION
# --------------------------------------------

# Set up macro for including draggable legend on map
# https://nbviewer.org/gist/talbertc-usgs/18f8901fc98f109f2b71156cf3ac81cd
template = """
{% macro html(this, kwargs) %}

<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>jQuery UI Draggable - Default functionality</title>
  <link rel="stylesheet" href="//code.jquery.com/ui/1.12.1/themes/base/jquery-ui.css">

  <script src="https://code.jquery.com/jquery-1.12.4.js"></script>
  <script src="https://code.jquery.com/ui/1.12.1/jquery-ui.js"></script>
  
  <script>
  $( function() {
    $( "#maplegend" ).draggable({
                    start: function (event, ui) {
                        $(this).css({
                            right: "auto",
                            top: "auto",
                            bottom: "auto"
                        });
                    }
                });
});

  </script>
</head>
<body>

 
<div id='maplegend' class='maplegend' 
    style='position: absolute; z-index:9999; border:2px solid grey; background-color:rgba(255, 255, 255, 0.8);
     border-radius:6px; padding: 10px; font-size:14px; right: 20px; bottom: 20px;'>
     
<div class='legend-title'>Legend</div>
<div class='legend-scale'>
  <ul class='legend-labels'>
    <li><span style='background:aqua;'></span>Group 1</li>
    <li><span style='background:black;'></span>Group 2</li>
    <li><span style='background:orange;'></span>Separator</li>
    <li><span style='background:magenta;'></span>Illegal Edge</li>

  </ul>
</div>
</div>
 
</body>
</html>

<style type='text/css'>
  .maplegend .legend-title {
    text-align: left;
    margin-bottom: 5px;
    font-weight: bold;
    font-size: 90%;
    }
  .maplegend .legend-scale ul {
    margin: 0;
    margin-bottom: 5px;
    padding: 0;
    float: left;
    list-style: none;
    }
  .maplegend .legend-scale ul li {
    font-size: 80%;
    list-style: none;
    margin-left: 0;
    line-height: 18px;
    margin-bottom: 2px;
    }
  .maplegend ul.legend-labels li span {
    display: block;
    float: left;
    height: 16px;
    width: 30px;
    margin-right: 5px;
    margin-left: 0;
    border: 1px solid #999;
    }
  .maplegend .legend-source {
    font-size: 80%;
    color: #777;
    clear: both;
    }
  .maplegend a {
    color: #777;
    }
</style>
{% endmacro %}"""

macro = MacroElement()
macro._template = Template(template)

def visualize_map(G, group_1, group_2, sep_group, illegal_edges, split = '', bound1 = None, bound2 = None):
    lon, lat = active_zone_ll.centroid.coords[0]
    zoom_start_val = 11

    # Basemap
    m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

    # Bounding box around active zone = Red
    sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
    geo_j.add_to(m)

    # Plot OSM Results
    for tree_geom in geometries['geometry']:

        if tree_geom.area > min_area:

            # Plot Polygon
            sim_geo = gpd.GeoSeries(tree_geom)#.simplify(tolerance=0.001)
            geo_j = sim_geo.to_json()
            geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"color": "green", "width": 2})#orange"})
            geo_j.add_to(m)
            # polygons.append(tree_geom)
    
    # Plot separator range
    if bound1 is not None and bound2 is not None:
        if split in ['vertical','longitude']:
            folium.PolyLine([[active_zone_bounds[1][0],bound2],[active_zone_bounds[1][1],bound2]], dash_array = 10, color = 'orange').add_to(m)
            folium.PolyLine([[active_zone_bounds[1][0],bound1],[active_zone_bounds[1][1],bound1]], dash_array = 10, color = 'orange').add_to(m)
        elif split in ['horizontal','latitude']:
            folium.PolyLine([[bound2,active_zone_bounds[0][0]],[bound2,active_zone_bounds[0][1]]], dash_array = 10, color = 'orange').add_to(m)
            folium.PolyLine([[bound1,active_zone_bounds[0][0]],[bound1,active_zone_bounds[0][1]]], dash_array = 10, color = 'orange').add_to(m)

    # Plot edges
    for edge in G.edges():
        c1 = centroids[edge[0]]
        c2 = centroids[edge[1]]
        folium.PolyLine([[c1.y,c1.x],[c2.y,c2.x]], weight=1, opacity = 0.6).add_to(m) #color='#808080'
    
    # Plot nodes
    for node in G.nodes():
        if node in group_1:
            color = 'aqua'#'#17bebb'
        elif node in group_2:
            color = 'black'#'#2a7de1'
        elif node in sep_group:
            color = 'orange'#'#f37820'
        else:
            color = 'blue'

        folium.CircleMarker(location=[centroids[node].y, centroids[node].x],
                                radius=3,#2.5,
                                color = color).add_to(m)#weight=5, color = 'green').add_to(m)
    # Highlight illegal edges
    for edge in illegal_edges:
        c1 = centroids[edge[0]]
        c2 = centroids[edge[1]]
        folium.PolyLine([[c1.y,c1.x],[c2.y,c2.x]], weight=4, color='magenta').add_to(m)
    
    # Add draggable legend
    m.get_root().add_child(macro)
    
    return m

## Map Setup

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))

active_zone_bounds

In [ ]:
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
min_area = 1e-6

tags = {'landuse': ['forest'], 'natural': ['wood']} #

geometries = ox.geometries_from_bbox(active_zone_bounds[1][0], active_zone_bounds[1][1], 
                                     active_zone_bounds[0][0], active_zone_bounds[0][1], tags=tags)

geometries.reset_index(inplace = True)

## Graph Setup

In [ ]:
with open('../data/fire_centroids_3.pkl', 'rb') as handle:
    centroids = pickle.load(handle)

with open('../data/fire_graph_3.pkl', 'rb') as handle:
    G = pickle.load(handle)

In [ ]:
# Identify and remove isolated nodes
isolated = list(nx.isolates(G))
print("Isolated nodes:", isolated)

# Remove corresponding centroids and nodes
for i in sorted(isolated)[::-1]:
    centroids.pop(i)
G.remove_nodes_from(isolated)
print("Nodes removed")
# Re-label nodes to properly work with D-Wave functions
G = nx.convert_node_labels_to_integers(G, ordering='sorted')

In [ ]:
print(len(centroids),len(G.nodes()))

In [ ]:
# Show networkx graph with coordinate positions
# positions = {}
# for i in range(len(centroids)):
#     centroid = centroids[i]
#     positions[i] = (centroid.x, centroid.y)
positions = [(centroid.x, centroid.y) for centroid in centroids]
print("Nodes:",len(G.nodes), "\tEdges:",len(G.edges))
nx.draw(G, positions, node_size = 5)

In [ ]:
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11
min_area = 1e-6
polygons = []
centroids = []

# Basemap
m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, height=500)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

# Plot OSM Results
for tree_geom in geometries['geometry']:
    if tree_geom.area > 0:
        # Plot Polygon
        sim_geo = gpd.GeoSeries(tree_geom)#.simplify(tolerance=0.001)
        geo_j = sim_geo.to_json()
        geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"color": "litegreen", "width": 1})#orange"})
        geo_j.add_to(m)
        polygons.append(tree_geom)

for node in G.nodes():

    folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]], radius=0.1,
                        color = 'green', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    
m

## D-Wave API Token

In [ ]:
my_token = '' ## Insert DWave Token Here

# Vertical Split (Separate East from West)

## Find "Minimal" Separator Number, Equally-sized Groups

In [ ]:
tolerance = 'default'

In [ ]:
# Build model with separation constraint
cqm = build_cqm(G)

# This is the part that uses up time on Leap.
# Run optimization.
sampler = LeapHybridCQMSampler(token=my_token)
sample = run_cqm_and_collect_solutions(cqm, sampler)

# Process results into classification groups and illegal edges
group_1, group_2, sep_group, illegal_edges = process_sample(G, sample)

visualize_results(G, group_1, group_2, sep_group, illegal_edges)

In [ ]:
G.edges[(0,1)]

In [ ]:
visualize_map(G, group_1, group_2, sep_group, illegal_edges)

## Find "Minimal" Separator Number, Group Size Tolerance

In [ ]:
# Build model with separation constraint
cqm = build_cqm_tol(G, tolerance = 80)

# This is the part that uses up time on Leap.
# Run optimization.
sampler = LeapHybridCQMSampler(token=my_token)
sample = run_cqm_and_collect_solutions(cqm, sampler)

# Process results into classification groups and illegal edges
group_1, group_2, sep_group, illegal_edges = process_sample(G, sample)

visualize_results(G, group_1, group_2, sep_group, illegal_edges)

## Find "Minimal" Separator Number, Group Size Tolerance

In [ ]:
#results = {}

# Build model with separation constraint
#tolerance_values = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 225, 250, 275, 300, 350, 400, 450, 500, 600, 700, 800, 1000]
tolerance_values = [280,290,310,320,330,340,360,370,380,390]



for tolerance_value in tolerance_values:
    cqm = build_cqm_tol(G, tolerance = tolerance_value)

    # This is the part that uses up time on Leap.
    # Run optimization.
    sampler = LeapHybridCQMSampler(token=my_token)
    sample = run_cqm_and_collect_solutions(cqm, sampler)

    # Process results into classification groups and illegal edges
    group_1, group_2, sep_group, illegal_edges = process_sample(G, sample)

    #visualize_results(G, group_1, group_2, sep_group, illegal_edges)

    results.update({tolerance_value: {'sample': sample, 'group_1':group_1, 'group_2':group_2, 'sep_group': sep_group, 'illegal_edges': illegal_edges} })


In [ ]:
keys = list(results.keys())
keys.sort()
keys

In [ ]:
import pickle 
import datetime

with open('../data/results/forest_separation_varying_tolerance_{}_2.pkl'.format(datetime.date.today()), 'wb') as f:
    pickle.dump(results, f)

In [ ]:
sep_group_sav = sep_group
group_1_sav = group_1
group_2_sav = group_2

# sep_group = sep_group_sav
# group_1 = group_1_sav
# group_2 = group_2_sav

In [ ]:
for node in sep_group:
    connected_to = []

    for edges in G.edges(node):
        if edges[0] == node:
            if edges[1] in group_1:
                connected_to.append('group_1')
            elif edges[1] in group_2:
                connected_to.append('group_2')
        if edges[1] == node:
            if edges[0] in group_1:
                connected_to.append('group_1')
            elif edges[0] in group_2:
                connected_to.append('group_2')

    if not ('group_1' in connected_to and 'group_2' in connected_to):
        print(node, connected_to)
        if connected_to[0] == 'group_1':
            group_1.append(node)
            sep_group.remove(node)
            print('added to group 1') 

        if connected_to[0] == 'group_2':
            group_2.append(node)
            sep_group.remove(node)
            print('added to group 2')  
            

In [ ]:
len(group_1), len(group_2), len(sep_group)

In [ ]:
process_sample_with_sep_preassigned(G, sample, sep_group, group_1, group_2)

In [ ]:
# H = nx.subgraph(G, group_1)
# nx.draw(G, with_labels=True, font_weight='bold', node_color='lightblue', node_size=500)

In [ ]:
# import copy
# H= copy.deepcopy(G)

# url = r'https://epqs.nationalmap.gov/v1/json?'

# def elevation_function(G):
#     """Query service using lat, lon. add the elevation values as a new column."""
#     elevations = []
#     for node in G.nodes:
                
#         # define rest query params
#         params = {
#             'output': 'json',
#             'x': G.nodes[node]['pos'][0],
#             'y': G.nodes[node]['pos'][1],
#             'units': 'Meters'}
        
#         # format query string and return query value
#         result = requests.get((url + urllib.parse.urlencode(params)))
#         #elevations.append(result.json()['USGS_Elevation_Point_Query_Service']['Elevation_Query']['Elevation'])
#         #new 2023:
#         G.nodes[node]['elevation'] = result.json()['value']

#     return G

# G = elevation_function(H)


In [ ]:
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

for edge in G.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)
    
# 
    # if node in group_1:
    #     folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
    #                         radius=0.1,
    #                         color = 'aqua', tooltip= str(node)).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    # elif node in group_2:
    #     folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
    #                         radius=0.1,
    #                         color = 'blue', tooltip= str(node)).add_to(m)  #weight=5, color = 'green').add_to(m)
    # else:
    #     folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
    #                         radius=0.1,
    #                         color = 'red', tooltip= str(node)).add_to(m)  #weight=5, color = 'green').add_to(m)



m

In [ ]:
G.nodes[0]

In [ ]:
float(G.nodes[0]['elevation_meters'])

In [ ]:
Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
largest_connected_component = Gcc[0]
G0 = G.subgraph(largest_connected_component)

In [ ]:
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 9

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

# for edge in G.edges():
#     c1 = G.nodes[edge[0]]['pos']
#     c2 = G.nodes[edge[1]]['pos']
#     folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)


for node in G0.nodes:
    if float(G.nodes[node]['elevation_meters']) < 100:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'darkgray', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    elif float(G.nodes[node]['elevation_meters']) < 200:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'purple', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)

    elif float(G.nodes[node]['elevation_meters']) < 300:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'cadetblue', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)


    elif float(G.nodes[node]['elevation_meters']) < 400:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'green', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)


    elif float(G.nodes[node]['elevation_meters']) < 500:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'lightgreen', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)


    elif float(G.nodes[node]['elevation_meters']) < 600:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'gold', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)


    elif float(G.nodes[node]['elevation_meters']) < 700:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color ='orange' , tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)


    elif float(G.nodes[node]['elevation_meters']) < 800:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'red', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)

    elif float(G.nodes[node]['elevation_meters']) > 800:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'darkred', tooltip= str(node) + ' ' + str(G.nodes[node]['elevation_meters']) ).add_to(m)  #weight=5, color = 'green').add_to(m)



m

In [ ]:
ridge_line = [1641, 1560, 1537, 1506, 1507, 1476, 1445, 1334, 1302, 1275]#, 540, 374, 256, 194, 130]

In [ ]:
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

for edge in G0.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)

for node in G0.nodes():

    if node in ridge_line:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'red', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    else:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'blue', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)


m

In [ ]:
print(active_zone_ll)

In [ ]:
separation_polygon = []

for point in ridge_line:
    separation_polygon.append([G.nodes[point]['pos'][1], G.nodes[point]['pos'][0]])

# separation_polygon.extend([[37.092, -121.79], [37.1129, -121.835], [37.1129, -121.835], [37.10, -121.85], [37.1048, -121.855], [37.113, -121.855], 
#                            [37.1129, -121.8725], [37.129, -121.875],  [37.135, -121.895], [37.151, -121.9], ## old/new break
#                            [37.151, -121.90], [37.151, -121.915], [37.135, -121.93], [37.15, -121.953], [37.156, -121.953], [37.163,-121.948], [37.163,-121.963], [37.19,-121.963], [37.19,-121.99],
#                            [37.202,-121.995], [37.196,-122.005], [37.25,-122],[37.25,-121.75], [37.046,-121.749]])

separation_polygon.insert(7,[37.065,-121.8])

separation_polygon.extend([[37.092, -121.79], [37.1129, -121.835], [37.1129, -121.835], [37.10, -121.85], [37.1048, -121.855], [37.113, -121.855], 
                           [37.1129, -121.8725], [37.129, -121.875], [37.129, -121.885],  [37.135, -121.89], [37.151, -121.9], ## old/new break
                           [37.151, -121.90], [37.151, -121.915], [37.169, -121.93], [37.169, -121.9475],  # [37.169, -121.9475], 
                           [37.179, -121.9475], [37.179, -121.9625], 
                           [37.19,-121.963], [37.19,-121.99],
                           [37.202,-121.995], [37.196,-122], [37.25,-122.001],[37.25,-121.75], [37.046,-121.749]])


separation_polygon.append(separation_polygon[0])

new_separation_polygon = []

for point in separation_polygon:
    new_separation_polygon.append([point[1], point[0]])

separation_polygon = Polygon(new_separation_polygon)


In [ ]:
[[coord[1], coord[0]] for coord in separation_polygon.exterior.coords]

In [ ]:
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()

folium.PolyLine([[coord[1], coord[0]] for coord in separation_polygon.exterior.coords], weight=3, color = 'darkorange').add_to(m)


geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

# sim_geo = gpd.GeoSeries(separation_polygon).simplify(tolerance=0.001)
# geo_j = sim_geo.to_json()
# geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "purple"})#orange"})
# geo_j.add_to(m)

n_teal = 0
n_blue = 0
n_red = 0

group_1 = []
group_2 = []


for edge in G0.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)

for node in G0.nodes():

    if node in ridge_line:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=4,
                            color = 'black', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.7,
                            color = 'darkorange', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
        n_red+=1
    elif separation_polygon.contains(Point(G.nodes[node]['pos'][0], G.nodes[node]['pos'][1])):
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'aqua', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
        group_1.append(node)
        n_teal+=1
        
    else: 
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'blue', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

        group_2.append(node)
        n_blue+=1
        

print('separator: {}, group 1: {}, group 2: {}'.format(n_red,n_teal, n_blue))
m

In [ ]:
illegal_edges = []

for u, v in G.edges:
    if u in group_1:
        xu0 = 1
        xu1 = 0
    elif u in group_2:
        xu0 = 0
        xu1 = 1 
    else:
        xu0 = 0
        xu1 = 0 
        
    if v in group_1:
        xv0 = 1
        xv1 = 0
    elif v in group_2:
        xv0 = 0
        xv1 = 1 
    else:
        xv0 = 0
        xv1 = 0 
        
    if xu0*xv1 == 1 or xu1*xv0 == 1:
        illegal_edges.append((u,v))
        print(u,v)
        print(xu0,xv1, xv0,xu1)
            
print("\nNumber of illegal edges:\t", len(illegal_edges))


In [ ]:
ride_separation_results = {'sep_group': ridge_line, 'group_1':group_1, 'group_2':group_2,  
                           'illegal_edges': illegal_edges, 'separation_polygon': separation_polygon}


with open('../data/results/forest_separation_ridge_line_{}.pkl'.format(datetime.date.today()), 'wb') as f:
    pickle.dump(ride_separation_results, f)

In [ ]:
G.nodes[1302]

## Optimization

In [ ]:
# Set up separator bounds and preassignment
diff = active_zone_bounds[0][1] - active_zone_bounds[0][0]
bound1 = active_zone_bounds[0][0]+diff/10
bound2 = active_zone_bounds[0][1]-diff/10

pre_G1, pre_G2 = preassign(G, 'vertical', bound1, bound2)
print(len(G.nodes),len(pre_G1),len(pre_G2))

In [ ]:
# !!! Warning !!! This cell uses Hybrid solver time on Leap. 
# You must use your own API token from Leap for my_token.

# Initialize dictionary for saving results and list for saving hubness counts
fire_results_dict = {}
fire_hubness_list = [0 for i in range(len(G.nodes()))]
max_sep = 0
num_trials = 3
complete_separation = False

# Iterate through size of separator class
for sep_size in range(1,31):
    fire_results_dict[sep_size] = {}
    for i in range(num_trials):
        print(f"Separator constraint = {sep_size} - Trial {i}\n")

        # Build model with separation constraint
        cqm = build_cqm_preassigned(G, size=sep_size, pre_G1=pre_G1, pre_G2=pre_G2)

        # This is the part that uses up time on Leap.
        # Run optimization.
        sampler = LeapHybridCQMSampler(token=my_token)
        sample = run_cqm_and_collect_solutions(cqm, sampler)

        # Process results into classification groups and illegal edges
        group_1, group_2, sep_group, illegal_edges = process_sample_preassigned(G, sample, pre_G1, pre_G2)

        visualize_results(G, group_1, group_2, sep_group, illegal_edges)
        print("-"*75+"\n\n\n\n")

        # Save results
        # unhappiness = (# illegal edges) + (# separator nodes off from target separator constraint)
        result = {
            'sep_group': sep_group,
            'illegal_edges': illegal_edges,
            'unhappiness': len(illegal_edges) + abs(sep_size - len(sep_group)),
            # 'unhappiness_neighbor_path': unhappiness_neighbor_path(G, group_1, group_2, sep_group, illegal_edges),
            # 'unhappiness_long_hop': unhappiness_long_hop(G, lower, higher, sep_group, illegal_edges),
            'group_1': group_1,
            'group_2': group_2,
            'bound1': bound1,
            'bound2': bound2,
            'split': 'vertical',
        }
        fire_results_dict[sep_size][i] = deepcopy(result)
        # Update hubness
        for node in sep_group:
            fire_hubness_list[node] += 1
        
        if len(illegal_edges) == 0:
            complete_separation = True
            best_trial = i
    
    max_sep = sep_size
    
    # Quit loop if find compelete separation
    if complete_separation:
        print("Complete separation, no illegal edges. Stopping loop.")
        break

In [ ]:
visualize_map(G, group_1, group_2, sep_group, illegal_edges, split='vertical', bound1=bound1, bound2=bound2)

In [ ]:
# Save results
# !!! Warning !!! Be careful not to accidentally overwrite an existing pickle file

with open("fire_results_dict_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_results_dict, fp)

with open("fire_hubness_list_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_hubness_list, fp)

# Sort nodes in descending order of hubness and save
fire_sorted_hubness_nodes = [x for _,x in sorted(zip(fire_hubness_list,list(range(len(G.nodes)))), reverse=True)]
with open("fire_sorted_hubness_nodes_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_sorted_hubness_nodes, fp)

In [ ]:

pickle.dump(G, open('../data/fire_graph_4.pkl', 'wb'))

## Hubness

In [ ]:
# !!! Warning !!! This cell uses time on Leap. 
# You must use your own API token for my_token.

fire_hubness_results_dict = {}

# Iterate through size of separator class
for sep_size in fire_results_dict.keys():
    fire_hubness_results_dict[sep_size] = {}
    for i in range(num_trials):
        separators = fire_sorted_hubness_nodes[0:sep_size]

        print(f"Separator size = {sep_size} - Trial {i}\n")

        # Build model with separation constraint
        cqm = build_cqm_from_sep_preassigned(G, separators, pre_G1, pre_G2)

        # This is the part that uses up time on Leap.
        # Run optimization.
        sampler = LeapHybridCQMSampler(token=my_token)
        sample = run_cqm_and_collect_solutions(cqm, sampler)

        # Process results into classification groups and illegal edges
        group_1, group_2, illegal_edges = process_sample_with_sep_preassigned(G, sample, separators, pre_G1, pre_G2)

        visualize_results(G, group_1, group_2, separators, illegal_edges)
        print("-"*75+"\n\n\n\n")

        # Save results
        # unhappiness = (# illegal edges) + (# separator nodes off from target separator constraint)
        result = {
            'sep_group': separators,
            'illegal_edges': illegal_edges,
            'unhappiness': len(illegal_edges) + abs(sep_size - len(separators)),
            # 'unhappiness_neighbor_path': unhappiness_neighbor_path(G, group_1, group_2, separators, illegal_edges),
            # 'unhappiness_long_hop': unhappiness_long_hop(G, lower, higher, separators, illegal_edges),
            'group_1': group_1,
            'group_2': group_2,
            'bound1': bound1,
            'bound2': bound2,
            'split': 'vertical',
        }
        fire_hubness_results_dict[sep_size][i] = deepcopy(result)

In [ ]:
visualize_map(G, group_1, group_2, sep_group, illegal_edges, split='vertical', bound1=bound1, bound2=bound2)

In [ ]:
# Save results
# !!! Warning !!! Be careful not to accidentally overwrite an existing pickle file

with open("fire_hubness_results_dict_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_hubness_results_dict, fp)

## Random (from largest optimal set)

In [ ]:
# !!! Warning !!! This cell uses time on Leap. 
# You must use your own API token for my_token.

fire_random_results_dict = {}

# max_sep = max(fire_results_dict.keys())
# for i in range(num_trials):
#     if len(fire_results_dict[max_sep][i]['illegal_edges']) == 0:
#         best_trial = i
#         break
last_separators = fire_results_dict[max_sep][best_trial]['sep_group']

# Iterate through size of separator class
for sep_size in fire_results_dict.keys():
    fire_random_results_dict[sep_size] = {}
    for i in range(num_trials):

        # From last optimization separators group, randomly sample sep_size number of separators
        separators = random.sample(last_separators, sep_size)

        print(f"Separator size = {sep_size} - Trial {i}\n")

        # Build model with separation constraint
        cqm = build_cqm_from_sep_preassigned(G, separators, pre_G1, pre_G2)

        # This is the part that uses up time on Leap.
        # Run optimization.
        sampler = LeapHybridCQMSampler(token=my_token)
        sample = run_cqm_and_collect_solutions(cqm, sampler)

        # Process results into classification groups and illegal edges
        group_1, group_2, illegal_edges = process_sample_with_sep_preassigned(G, sample, separators, pre_G1, pre_G2)

        visualize_results(G, group_1, group_2, separators, illegal_edges)
        print("-"*75+"\n\n\n\n")

        # Save results
        # unhappiness = (# illegal edges) + (# nodes could be moved to get almost-equal groups) + (# separator nodes off from target separator constraint)
        result = {
            'sep_group': separators,
            'illegal_edges': illegal_edges,
            'unhappiness': len(illegal_edges) + abs(sep_size - len(separators)),
            # 'unhappiness_neighbor_path': unhappiness_neighbor_path(G, group_1, group_2, separators, illegal_edges),
            # 'unhappiness_long_hop': unhappiness_long_hop(G, lower, higher, separators, illegal_edges),
            'group_1': group_1,
            'group_2': group_2,
            'bound1': bound1,
            'bound2': bound2,
            'split': 'vertical',
        }
        fire_random_results_dict[sep_size][i] = deepcopy(result)

In [ ]:
# Save results
# !!! Warning !!! Be careful not to accidentally overwrite an existing pickle file

with open("fire_random_results_dict_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_random_results_dict, fp)

## Unhappiness Graph

In [ ]:
# PLOT UNHAPPINESS
x = list(fire_results_dict.keys())
# x0 = [len(fire_results_dict[i]['sep_group']) for i in x]
alpha = 0.7
dwave_unhappiness = [min([len(fire_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]
hubness_unhappiness = [min([len(fire_hubness_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]
random_unhappiness = [min([len(fire_random_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]

plt.scatter(x, dwave_unhappiness, label = 'Optimal solution for each number', alpha = alpha, color = 'darkturquoise')
plt.scatter(x, hubness_unhappiness, label = 'Adding separators in order of hubness', alpha = alpha, color = 'goldenrod')
plt.scatter(x, random_unhappiness, label = f'Randomly selected from {max_sep} optimal set', alpha = alpha, color = 'sienna')

plt.legend()
# plt.yscale('log')
xlab = plt.xlabel("Number of Separators")
ylab = plt.ylabel("Number of Illegal Edges/\nFire Spread")
xlab.set_style('italic')
xlab.set_size(10)
ylab.set_style('italic')
ylab.set_size(10)
plt.title(f"Comparison of Implementation Strategy - Best of {num_trials} Trials\nD-Wave - California Forest")
plt.grid()
# plt.savefig("fire_unhappiness.png")
plt.show()

## Tinkering With Hubness

Try using nodes from best separator group in order of hubness, rather than selecting from all nodes in order of hubness.

In [ ]:
# !!! Warning !!! This cell uses time on Leap. 
# You must use your own API token for my_token.

fire_opt_hubness_results_dict = {}
# Sort nodes from best separator group in order of hubness
fire_opt_sorted_hubness_nodes = [x for _,x in sorted(zip([fire_hubness_list[i] for i in fire_results_dict[max_sep][best_trial]['sep_group']],fire_results_dict[max_sep][best_trial]['sep_group']), reverse=True)]
# Iterate through size of separator class
for sep_size in fire_results_dict.keys():
    fire_opt_hubness_results_dict[sep_size] = {}
    for i in range(num_trials):
        separators = fire_opt_sorted_hubness_nodes[0:sep_size]

        print(f"Separator size = {sep_size} - Trial {i}\n")

        # Build model with separation constraint
        cqm = build_cqm_from_sep_preassigned(G, separators, pre_G1, pre_G2)

        # This is the part that uses up time on Leap.
        # Run optimization.
        sampler = LeapHybridCQMSampler(token=my_token)
        sample = run_cqm_and_collect_solutions(cqm, sampler)

        # Process results into classification groups and illegal edges
        group_1, group_2, illegal_edges = process_sample_with_sep_preassigned(G, sample, separators, pre_G1, pre_G2)

        visualize_results(G, group_1, group_2, separators, illegal_edges)
        print("-"*75+"\n\n\n\n")

        # Save results
        # unhappiness = (# illegal edges) + (# separator nodes off from target separator constraint)
        result = {
            'sep_group': separators,
            'illegal_edges': illegal_edges,
            'unhappiness': len(illegal_edges) + abs(sep_size - len(separators)),
            # 'unhappiness_neighbor_path': unhappiness_neighbor_path(G, group_1, group_2, separators, illegal_edges),
            # 'unhappiness_long_hop': unhappiness_long_hop(G, lower, higher, separators, illegal_edges),
            'group_1': group_1,
            'group_2': group_2,
            'bound1': bound1,
            'bound2': bound2,
            'split': 'vertical',
        }
        fire_opt_hubness_results_dict[sep_size][i] = deepcopy(result)

In [ ]:
# Save results
# !!! Warning !!! Be careful not to accidentally overwrite an existing pickle file

with open("fire_opt_hubness_results_dict_1-3-2024.pkl", "wb") as fp:   #Pickling
    pickle.dump(fire_opt_hubness_results_dict, fp)

In [ ]:
# PLOT UNHAPPINESS
x = list(fire_results_dict.keys())
# x0 = [len(fire_results_dict[i]['sep_group']) for i in x]
alpha = 0.7
dwave_unhappiness = [min([len(fire_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]
hubness_unhappiness = [min([len(fire_hubness_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]
random_unhappiness = [min([len(fire_random_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]
opt_hubness_unhappiness = [min([len(fire_opt_hubness_results_dict[i][j]['illegal_edges']) for j in range(num_trials)]) for i in x]

plt.scatter(x, dwave_unhappiness, label = 'Optimal solution for each number', alpha = alpha, color = 'darkturquoise')
plt.scatter(x, hubness_unhappiness, label = 'Adding separators in order of hubness', alpha = alpha, color = 'goldenrod')
plt.scatter(x, opt_hubness_unhappiness, label = f'Adding separators from {max_sep} optimal set\nin order of hubness', alpha = alpha, color = 'red')
plt.scatter(x, random_unhappiness, label = f'Randomly selected from {max_sep} optimal set', alpha = alpha, color = 'sienna')

plt.legend()
# plt.yscale('log')
xlab = plt.xlabel("Number of Separators")
ylab = plt.ylabel("Number of Illegal Edges/\nFire Spread")
xlab.set_style('italic')
xlab.set_size(10)
ylab.set_style('italic')
ylab.set_size(10)
plt.title(f"Comparison of Implementation Strategy - Best of {num_trials} Trials\nD-Wave - California Forest")
plt.grid()
# plt.savefig("fire_unhappiness.png")
plt.show()